# Square-grid directional experiments

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import gc
import io
import json
import random
import re
import shutil

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torch_geometric.nn import GCNConv, GraphConv, RGCNConv, SAGEConv

# Experiment settings for 2022-06-17
ALL_SPLITS = [42, 123, 2026, 3407, 777]
RUN_SPLITS = ALL_SPLITS
DATE = "2022-06-17"
GROUP = "public"
MLP_MODELS = ["MLP_center", "MLP_ordered"]
GNN_MODELS = ["GCN", "GraphSAGE", "GraphConv", "RGCN_directional"]
INIT_SEEDS = [11, 22, 33]

N_TRIALS = 20
TUNING_INIT_SEED = 11
TUNE_MAX_EPOCHS = 1500
DROPOUT = 0.1
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]
DIRECTIONS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
MLP_OUT = WORK / "results_grid90_ordered_mlp_v1"
GNN_OUT = WORK / "results_grid90_four_gnn_v1"
MLP_OUT.mkdir(parents=True, exist_ok=True)
GNN_OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    print("GPU:", torch.cuda.get_device_name(0))
optuna.logging.set_verbosity(optuna.logging.WARNING)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# Load 2022 arrays from extracted folder or zip archive
data_rel = f"{DATE}/{GROUP}/mlp_inputs_raw.npz"
split_rels = {s: f"splits_80_10_10/split_{s}.npz" for s in ALL_SPLITS}

npz_matches = [p for p in INPUT_ROOT.rglob("mlp_inputs_raw.npz") if p.as_posix().endswith("/" + data_rel)]
if npz_matches:
    root = npz_matches[0].parents[2]
    payload = {rel: (root / rel).read_bytes() for rel in [data_rel, *split_rels.values()]}
else:
    zips = list(INPUT_ROOT.rglob("mlp_ordered_grid90_20220617_public_v1.zip"))
    if not zips:
        raise FileNotFoundError("Could not find 2022 MLP input files")
    with ZipFile(zips[0]) as zf:
        payload = {rel: zf.read(rel) for rel in [data_rel, *split_rels.values()]}

with np.load(io.BytesIO(payload[data_rel])) as z:
    X6 = z["X_center"].astype("float32")
    X62 = z["X_ordered"].astype("float32")
    has_neighbor = z["neighbor_mask"]
    Y = z["y_lst_C"].astype("float32")
    GRID_IDS = z["grid_id"]

n = len(Y)
neighbors_raw = X62[:, 6:54].reshape(n, 8, 6)

splits = {}
for seed, rel in split_rels.items():
    with np.load(io.BytesIO(payload[rel])) as z:
        masks = {k: z[k] for k in ("train_mask", "val_mask", "test_mask", "trainval_mask")}
    assert np.all(masks["train_mask"].astype(int) + masks["val_mask"].astype(int) + masks["test_mask"].astype(int) == 1)
    splits[seed] = masks

print(f"Loaded 2022 data: {n} nodes")

# Fit scaler on fit_mask only and prepare center or ordered inputs
def make_data(model_name, fit_mask):
    fit_x = X6[fit_mask].astype("float64")
    x_mean = fit_x.mean(axis=0).astype("float32")
    x_std = fit_x.std(axis=0, ddof=0).astype("float32")
    x_std[x_std < 1e-8] = 1.0

    center = (X6 - x_mean) / x_std
    if model_name == "MLP_center":
        x = center
    else:
        nb_x = (neighbors_raw - x_mean) / x_std
        nb_x[~has_neighbor] = 0.0
        x = np.concatenate([center, nb_x.reshape(n, 48), has_neighbor.astype("float32")], axis=1)

    fit_y = Y[fit_mask].astype("float64")
    y_mean, y_std = float(fit_y.mean()), float(fit_y.std(ddof=0))
    stats = {
        "features": FEATURES, "x_mean": x_mean.tolist(), "x_std": x_std.tolist(),
        "y_mean": y_mean, "y_std": y_std, "number_of_fit_nodes": int(fit_mask.sum()),
    }
    xt = torch.as_tensor(x, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor((Y - y_mean) / y_std, dtype=torch.float32, device=DEVICE)
    return xt, yt, stats

def make_mlp(input_dim, params):
    acts = {"relu": nn.ReLU, "elu": nn.ELU, "gelu": nn.GELU}
    layers, width = [], input_dim
    for _ in range(params["n_layers"]):
        layers += [nn.Linear(width, params["hidden_dim"]), acts[params["activation"]](), nn.Dropout(DROPOUT)]
        width = params["hidden_dim"]
    layers.append(nn.Linear(width, 1))
    return nn.Sequential(*layers).to(DEVICE)

def suggest_params(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "n_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
        "activation": trial.suggest_categorical("activation", ["relu", "elu", "gelu"]),
        "patience": trial.suggest_categorical("patience", [50, 100, 150]),
    }

def select_rows(tensor, mask):
    return tensor.index_select(0, torch.as_tensor(np.flatnonzero(mask), device=DEVICE))

def mlp_step(model, optimizer, x, y):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(x).squeeze(-1), y)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    return loss.detach()

## Tune, refit, and evaluate 2022 MLPs

In [ ]:
def tune_mlp(model_name, seed, folder):
    folder.mkdir(parents=True, exist_ok=True)
    sel_file = folder / "best_selection.json"
    if sel_file.exists():
        return json.loads(sel_file.read_text())

    masks = splits[seed]
    x, y, stats = make_data(model_name, masks["train_mask"])
    xtr, ytr = select_rows(x, masks["train_mask"]), select_rows(y, masks["train_mask"])
    xva, yva = select_rows(x, masks["val_mask"]), select_rows(y, masks["val_mask"])
    save_json(folder / "scaler_tuning.json", stats)

    study = optuna.create_study(
        direction="minimize", study_name="validation_rmse_C",
        storage=f"sqlite:///{folder / 'optuna.sqlite3'}",
        sampler=optuna.samplers.TPESampler(seed=seed), load_if_exists=True,
    )

    def objective(trial):
        params = suggest_params(trial)
        seed_everything(TUNING_INIT_SEED)
        model = make_mlp(xtr.shape[1], params)
        optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)

        best_val, best_epoch = float("inf"), 0
        for epoch in range(1, TUNE_MAX_EPOCHS + 1):
            mlp_step(model, optimizer, xtr, ytr)
            model.eval()
            with torch.no_grad():
                rmse = (model(xva).squeeze(-1) - yva).square().mean().sqrt().item() * stats["y_std"]
            if rmse < best_val:
                best_val, best_epoch = rmse, epoch
            if epoch - best_epoch >= params["patience"]:
                break

        trial.set_user_attr("best_epoch", best_epoch)
        print(f"[Optuna] {model_name} | split={seed} | trial={trial.number} | val={best_val:.4f} C | best_epoch={best_epoch}")
        return best_val

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - done)

    best = study.best_trial
    selection = {
        "model": model_name, "split_seed": seed, "best_trial": best.number,
        "best_val_RMSE_C": best.value, "best_epoch": int(best.user_attrs["best_epoch"]),
        "params": best.params,
    }
    save_json(sel_file, selection)
    study.trials_dataframe().to_csv(folder / "optuna_trials.csv", index=False)
    return selection
#####################################################
def refit_mlp(model_name, seed, selection, folder):
    fit_mask = splits[seed]["trainval_mask"]
    x, y, stats = make_data(model_name, fit_mask)
    xfit, yfit = select_rows(x, fit_mask), select_rows(y, fit_mask)
    save_json(folder / "scaler_final.json", stats)

    params, epochs = selection["params"], selection["best_epoch"]
    for init_seed in INIT_SEEDS:
        run_dir = folder / f"init_{init_seed}"
        run_dir.mkdir(exist_ok=True)
        if (run_dir / "completed.json").exists() and (run_dir / "model.pt").exists():
            continue

        seed_everything(init_seed)
        model = make_mlp(xfit.shape[1], params)
        optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)
        losses = [mlp_step(model, optimizer, xfit, yfit).item() for _ in range(epochs)]

        torch.save(model.state_dict(), run_dir / "model.pt")
        pd.DataFrame({"epoch": np.arange(1, epochs + 1), "train_MSE_standardized": losses}).to_csv(run_dir / "history.csv", index=False)
        save_json(run_dir / "completed.json", {
            "model": model_name, "split_seed": seed, "init_seed": init_seed,
            "epochs": epochs, "input_dim": xfit.shape[1],
            "parameters": sum(p.numel() for p in model.parameters()),
        })

# Run tuning and refitting for 2022 MLPs
for seed in RUN_SPLITS:
    for name in MLP_MODELS:
        folder = MLP_OUT / f"split_{seed}" / name
        sel = tune_mlp(name, seed, folder)
        refit_mlp(name, seed, sel, folder)
        gc.collect()

# Evaluate 2022 MLPs on held-out test nodes
rows, selections = [], []
for seed in ALL_SPLITS:
    masks = splits[seed]
    for name in MLP_MODELS:
        folder = MLP_OUT / f"split_{seed}" / name
        sel = json.loads((folder / "best_selection.json").read_text())
        params = sel["params"]
        x, _, stats = make_data(name, masks["trainval_mask"])
        xtest = select_rows(x, masks["test_mask"])
        truth = Y[masks["test_mask"]].astype("float64")

        for init_seed in INIT_SEEDS:
            model = make_mlp(xtest.shape[1], params)
            model.load_state_dict(torch.load(folder / f"init_{init_seed}" / "model.pt", map_location=DEVICE, weights_only=True))
            model.eval()
            with torch.no_grad():
                pred_std = model(xtest).squeeze(-1).cpu().numpy().astype("float64")
            err = pred_std * stats["y_std"] + stats["y_mean"] - truth
            rows.append({
                "date": DATE, "feature_group": GROUP, "model": name,
                "split_seed": seed, "init_seed": init_seed, "subset": "test",
                "MAE": np.abs(err).mean(), "RMSE": np.sqrt(np.mean(err ** 2)),
                "R2": 1 - np.sum(err ** 2) / np.sum((truth - truth.mean()) ** 2),
                "Bias": err.mean(),
            })

        selections.append({
            "model": name, "split_seed": seed, "input_dim": xtest.shape[1],
            "parameters": sum(p.numel() for p in model.parameters()),
            "best_val_RMSE_C": sel["best_val_RMSE_C"], "final_epochs": sel["best_epoch"], **params,
        })

mlp_metrics = pd.DataFrame(rows)
measures = ["MAE", "RMSE", "R2", "Bias"]
mlp_per_split = mlp_metrics.groupby(["model", "split_seed"], as_index=False)[measures].mean()
mlp_summary = mlp_per_split.groupby("model")[measures].agg(["mean", "std"])
mlp_summary.columns = [f"{m}_{s}" for m, s in mlp_summary.columns]
mlp_summary = mlp_summary.reset_index()

paired = mlp_per_split.pivot(index="split_seed", columns="model", values="RMSE").reindex(ALL_SPLITS).reset_index()
paired.columns.name = None
paired["ordered_minus_center_C"] = paired["MLP_ordered"] - paired["MLP_center"]
delta = paired["ordered_minus_center_C"]
paired_summary = pd.DataFrame([{
    "delta_mean_C": delta.mean(), "delta_std_C": delta.std(ddof=1),
    "ordered_better_splits": int((delta < 0).sum()), "number_of_splits": len(delta),
}])

mlp_metrics.to_csv(MLP_OUT / "metrics_all_runs.csv", index=False)
mlp_per_split.to_csv(MLP_OUT / "metrics_per_split.csv", index=False)
mlp_summary.to_csv(MLP_OUT / "summary.csv", index=False)
paired.to_csv(MLP_OUT / "paired_rmse.csv", index=False)
paired_summary.to_csv(MLP_OUT / "paired_summary.csv", index=False)
pd.DataFrame(selections).to_csv(MLP_OUT / "best_params_all_splits.csv", index=False)

display(mlp_summary.round(4))
display(paired_summary.round(4))
shutil.make_archive(str(WORK / "results_grid90_ordered_mlp_v1_backup"), "zip", root_dir=MLP_OUT.parent, base_dir=MLP_OUT.name)

## Build 8-direction graph and run 2022 GNNs + comparisons

In [ ]:
# Build 8-direction edges from grid_id (format: r{row}_c{col})
offsets = [(-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1)]
grid_positions = []
for gid in GRID_IDS:
    m = re.fullmatch(r"r(\d+)_c(\d+)", str(gid))
    grid_positions.append((int(m[1]), int(m[2])))

pos_to_idx = {pos: i for i, pos in enumerate(grid_positions)}
sources, targets, relations = [], [], []
for rel_id, (dr, dc) in enumerate(offsets):
    nb_idx = np.array([pos_to_idx.get((r + dr, c + dc), -1) for r, c in grid_positions], dtype=np.int64)
    present = nb_idx >= 0
    centers = np.flatnonzero(present)
    sources.append(nb_idx[present])
    targets.append(centers)
    relations.append(np.full(len(centers), rel_id, dtype=np.int64))

EDGE = torch.as_tensor(np.stack([np.concatenate(sources), np.concatenate(targets)]), dtype=torch.long, device=DEVICE)
EDGE_TYPE = torch.as_tensor(np.concatenate(relations), dtype=torch.long, device=DEVICE)
####################################
class GridGNN(nn.Module):
    def __init__(self, name, params):
        super().__init__()
        self.name = name
        self.layers = nn.ModuleList()
        hidden, in_dim = params["hidden_dim"], 6
        for _ in range(params["n_layers"]):
            if name == "GCN":
                conv = GCNConv(in_dim, hidden, cached=True)
            elif name == "GraphSAGE":
                conv = SAGEConv(in_dim, hidden, aggr="mean")
            elif name == "GraphConv":
                conv = GraphConv(in_dim, hidden, aggr="add")
            elif name == "RGCN_directional":
                conv = RGCNConv(in_dim, hidden, num_relations=8, num_bases=2, aggr="add", root_weight=True)
            else:
                raise ValueError(name)
            self.layers.append(conv)
            in_dim = hidden

        self.act = {"relu": nn.ReLU, "elu": nn.ELU, "gelu": nn.GELU}[params["activation"]]()
        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        for conv in self.layers:
            x = conv(x, EDGE, EDGE_TYPE) if self.name == "RGCN_directional" else conv(x, EDGE)
            x = self.dropout(self.act(x))
        return self.head(x).squeeze(-1)

def gnn_step(model, optimizer, x, y, fit_mask):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(x)[fit_mask], y[fit_mask])
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    return loss.detach()

def tune_gnn(name, seed, folder):
    folder.mkdir(parents=True, exist_ok=True)
    sel_file = folder / "best_selection.json"
    if sel_file.exists():
        return json.loads(sel_file.read_text())

    masks = splits[seed]
    x, y, stats = make_data("MLP_center", masks["train_mask"])
    fit = torch.as_tensor(masks["train_mask"], dtype=torch.bool, device=DEVICE)
    val = torch.as_tensor(masks["val_mask"], dtype=torch.bool, device=DEVICE)
    save_json(folder / "scaler_tuning.json", stats)

    study = optuna.create_study(
        study_name="validation_rmse_C", direction="minimize",
        storage=f"sqlite:///{folder / 'optuna.sqlite3'}",
        sampler=optuna.samplers.TPESampler(seed=seed), load_if_exists=True,
    )

    def objective(trial):
        params = suggest_params(trial)
        seed_everything(TUNING_INIT_SEED)
        model = GridGNN(name, params).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)

        best_val, best_epoch = float("inf"), 0
        for epoch in range(1, TUNE_MAX_EPOCHS + 1):
            gnn_step(model, optimizer, x, y, fit)
            model.eval()
            with torch.no_grad():
                val_rmse = (model(x)[val] - y[val]).square().mean().sqrt().item() * stats["y_std"]
            if val_rmse < best_val:
                best_val, best_epoch = val_rmse, epoch
            if epoch - best_epoch >= params["patience"]:
                break

        trial.set_user_attr("best_epoch", best_epoch)
        print(f"[Optuna] {name} | split={seed} | trial={trial.number} | val={best_val:.4f} C | best_epoch={best_epoch}")
        return best_val

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - done)

    best = study.best_trial
    selected = {
        "model": name, "split_seed": seed, "best_trial": best.number,
        "best_val_RMSE_C": best.value, "best_epoch": int(best.user_attrs["best_epoch"]),
        "params": best.params,
    }
    study.trials_dataframe().to_csv(folder / "optuna_trials.csv", index=False)
    save_json(sel_file, selected)
    return selected

def refit_gnn(name, seed, folder, selected):
    fit_np = splits[seed]["trainval_mask"]
    x, y, stats = make_data("MLP_center", fit_np)
    fit = torch.as_tensor(fit_np, dtype=torch.bool, device=DEVICE)
    save_json(folder / "scaler_final.json", stats)

    params, epochs = selected["params"], selected["best_epoch"]
    for init in INIT_SEEDS:
        run = folder / f"init_{init}"
        run.mkdir(exist_ok=True)
        if (run / "completed.json").exists() and (run / "model.pt").exists():
            continue

        seed_everything(init)
        model = GridGNN(name, params).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)
        losses = [gnn_step(model, optimizer, x, y, fit).item() for _ in range(epochs)]

        torch.save(model.state_dict(), run / "model.pt")
        pd.DataFrame({"epoch": np.arange(1, epochs + 1), "train_MSE_standardized": losses}).to_csv(run / "history.csv", index=False)
        save_json(run / "completed.json", {
            "model": name, "split_seed": seed, "init_seed": init,
            "epochs": epochs, "parameters": sum(p.numel() for p in model.parameters()),
        })

# Run tuning and refitting for 2022 GNNs
for seed in RUN_SPLITS:
    for name in GNN_MODELS:
        folder = GNN_OUT / f"split_{seed}" / name
        sel = tune_gnn(name, seed, folder)
        refit_gnn(name, seed, folder, sel)
        gc.collect()

# Evaluate 2022 GNNs and compare with MLP_ordered and RGCN_directional
run_rows, param_rows = [], []
for seed in ALL_SPLITS:
    masks = splits[seed]
    x, _, stats = make_data("MLP_center", masks["trainval_mask"])
    test = torch.as_tensor(masks["test_mask"], dtype=torch.bool, device=DEVICE)
    truth = Y[masks["test_mask"]].astype("float64")

    for name in GNN_MODELS:
        folder = GNN_OUT / f"split_{seed}" / name
        sel = json.loads((folder / "best_selection.json").read_text())
        params = sel["params"]

        for init in INIT_SEEDS:
            model = GridGNN(name, params).to(DEVICE)
            model.load_state_dict(torch.load(folder / f"init_{init}" / "model.pt", map_location=DEVICE, weights_only=True))
            model.eval()
            with torch.no_grad():
                pred_std = model(x)[test].cpu().numpy().astype("float64")
            err = pred_std * stats["y_std"] + stats["y_mean"] - truth
            run_rows.append({
                "date": DATE, "feature_group": GROUP, "model": name,
                "split_seed": seed, "init_seed": init, "subset": "test",
                "MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt(np.mean(err ** 2))),
                "R2": float(1 - np.sum(err ** 2) / np.sum((truth - truth.mean()) ** 2)),
                "Bias": float(err.mean()),
            })

        param_rows.append({
            "model": name, "split_seed": seed, "parameters": sum(p.numel() for p in model.parameters()),
            "best_val_RMSE_C": sel["best_val_RMSE_C"], "final_epochs": sel["best_epoch"], **params,
        })

gnn_runs = pd.DataFrame(run_rows)
gnn_per_split = gnn_runs.groupby(["model", "split_seed"], as_index=False)[measures].mean()
gnn_runs.to_csv(GNN_OUT / "metrics_all_runs.csv", index=False)
gnn_per_split.to_csv(GNN_OUT / "metrics_per_split.csv", index=False)
pd.DataFrame(param_rows).to_csv(GNN_OUT / "best_params_all_splits.csv", index=False)

# Combine with MLP results and compute paired gains
combined = pd.concat([mlp_per_split, gnn_per_split], ignore_index=True)
summary = combined.groupby("model")[measures].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index().sort_values("RMSE_mean")
combined.to_csv(GNN_OUT / "comparison_per_split.csv", index=False)
summary.to_csv(GNN_OUT / "comparison_summary.csv", index=False)

wide = combined.pivot(index="split_seed", columns="model", values="RMSE").reindex(ALL_SPLITS)
deltas_vs_mlp = wide[GNN_MODELS].sub(wide["MLP_ordered"], axis=0)
cmp_mlp = pd.DataFrame({
    "delta_mean_C": deltas_vs_mlp.mean(), "delta_std_C": deltas_vs_mlp.std(ddof=1),
    "gnn_lower_rmse_splits": (deltas_vs_mlp < 0).sum(), "number_of_splits": len(deltas_vs_mlp),
})
deltas_vs_mlp.to_csv(GNN_OUT / "paired_vs_ordered_each_split.csv")
cmp_mlp.to_csv(GNN_OUT / "paired_vs_ordered_summary.csv")

# Directional gain: undirected GNNs minus RGCN_directional
gain = wide[["GraphSAGE", "GraphConv", "GCN"]].sub(wide["RGCN_directional"], axis=0)
gain_summary = pd.DataFrame({
    "mean_RMSE_reduction_C": gain.mean(), "SD_across_splits_C": gain.std(ddof=1),
    "splits_RGCN_better": (gain > 0).sum(), "number_of_splits": gain.count(),
}).rename_axis("comparison_model")
gain.to_csv(GNN_OUT / "directional_gain_per_split.csv")
gain_summary.to_csv(GNN_OUT / "directional_gain_summary.csv")

display(summary.round(4))
display(cmp_mlp.round(4))
display(gain_summary.round(4))
shutil.make_archive(str(GNN_OUT.parent / (GNN_OUT.name + "_backup")), "zip", root_dir=GNN_OUT.parent, base_dir=GNN_OUT.name)

## 2023 & 2024 setup, shared graph

In [ ]:
from functools import partial
import time
import yaml
from IPython.display import FileLink

from padova_core.models import DirectionalRegressor
from padova_core.neighbors import scale_directional_inputs

RUN_DATES = ["2023-08-15", "2024-08-25"]
SPLITS = [42, 123, 2026, 3407, 777]
INIT_SEEDS = [11, 22, 33]
MODELS = ["MLP_center", "MLP_ordered", "GCN", "GraphSAGE", "GraphConv", "RGCN_directional"]
FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]
DIRECTIONS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

N_TRIALS = 20
MAX_EPOCHS = 1500
TUNING_SEED = 11
DROPOUT = 0.1
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

PACKAGE = "grid90_2023_2024_public_training_v1"
WORK = Path("/kaggle/working")
candidates = [
    p.parent for p in Path("/kaggle/input").rglob("experiment_config.json")
    if p.parent.name == PACKAGE and (p.parent / "shared" / "neighbor_index.npy").exists()
]
if not candidates:
    raise FileNotFoundError(f"Could not find {PACKAGE}")

DATA = candidates[0]
SHARED = DATA / "shared"

# Load shared 8-direction graph and fixed splits
NB = np.load(SHARED / "neighbor_index.npy")
E = np.load(SHARED / "edge_index.npy")
R = np.load(SHARED / "edge_type.npy")
assert np.array_equal(NB[E[1], R], E[0]), "Edge types do not match neighbor slots."

EDGE = torch.as_tensor(E, dtype=torch.long, device=DEVICE)
REL = torch.as_tensor(R, dtype=torch.long, device=DEVICE)

MASKS = {}
for split in SPLITS:
    with np.load(SHARED / "splits" / f"split_{split}.npz") as f:
        m = {k: f[k].astype(bool) for k in f.files}
    assert np.all(m["train_mask"].astype(int) + m["val_mask"].astype(int) + m["test_mask"].astype(int) == 1)
    assert np.array_equal(m["final_train_mask"], m["train_mask"] | m["val_mask"])
    MASKS[split] = m

def load_date(date):
    folder = DATA / date
    x = np.load(folder / "X_center_raw.npy").astype(np.float32)
    y = np.load(folder / "y_lst_C.npy").astype(np.float32).reshape(-1)
    return x, y

def prepare(x, y, fit_mask):
    center, ordered, target, scaler = scale_directional_inputs(x, y, fit_mask, NB)
    return {
        "center": torch.as_tensor(center, dtype=torch.float32, device=DEVICE),
        "ordered": torch.as_tensor(ordered, dtype=torch.float32, device=DEVICE),
        "y": torch.as_tensor(target, dtype=torch.float32, device=DEVICE),
        "scaler": scaler, "ym": scaler["y_mean"], "ys": scaler["y_std"],
    }

def mask_idx(mask):
    return torch.as_tensor(np.flatnonzero(mask), dtype=torch.long, device=DEVICE)

LSTModel = partial(
    DirectionalRegressor, n_features=len(FEATURES), n_directions=NB.shape[1],
    edge_index=EDGE, edge_type=REL, dropout=DROPOUT,
)

def train_epoch(model, optimizer, data, fit_idx):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(data)[fit_idx], data["y"][fit_idx])
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    return loss.detach()

def tune_six(date, name, split, data, tr_idx, va_idx, folder):
    best_file = folder / "best_params.json"
    if best_file.exists():
        return json.loads(best_file.read_text())

    study = optuna.create_study(
        study_name=f"grid90_{date}_{name}_{split}",
        storage=f"sqlite:///{folder / 'optuna.db'}",
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=split), load_if_exists=True,
    )
    hist_dir = folder / "trial_histories"
    hist_dir.mkdir(exist_ok=True)

    def objective(trial):
        params = suggest_params(trial)
        seed_everything(TUNING_SEED)
        model = LSTModel(name, params).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)

        best_val, best_epoch, history = float("inf"), 0, []
        for epoch in range(1, MAX_EPOCHS + 1):
            loss = train_epoch(model, optimizer, data, tr_idx)
            model.eval()
            with torch.no_grad():
                val_rmse = (model(data)[va_idx] - data["y"][va_idx]).square().mean().sqrt().item() * data["ys"]
            history.append((epoch, loss.item(), val_rmse))
            if val_rmse < best_val:
                best_val, best_epoch = val_rmse, epoch
            if epoch - best_epoch >= params["patience"]:
                break

        trial.set_user_attr("best_epoch", best_epoch)
        pd.DataFrame(history, columns=["epoch", "train_MSE_standardized", "val_RMSE_C"]).to_csv(
            hist_dir / f"trial_{trial.number}.csv", index=False
        )
        print(f"[Optuna] {date} | {name} | split={split} | trial={trial.number} | val={best_val:.4f} C | best_epoch={best_epoch}")
        return best_val

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - done)

    study.trials_dataframe().to_csv(folder / "optuna_trials.csv", index=False)
    best = study.best_trial
    result = {
        "date": date, "model": name, "split_seed": split, "best_trial": best.number,
        "best_epoch": int(best.user_attrs["best_epoch"]), "best_val_RMSE_C": float(best.value),
        "params": best.params,
    }
    shutil.copy2(hist_dir / f"trial_{best.number}.csv", folder / "best_validation_history.csv")
    save_json(best_file, result)
    return result

def final_fit_six(date, name, split, seed, best, data, fit_idx, folder):
    run = folder / f"init_{seed}"
    run.mkdir(exist_ok=True)
    if (run / "completed.json").exists() and (run / "model.pt").exists():
        return

    seed_everything(seed)
    model = LSTModel(name, best["params"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best["params"]["learning_rate"], weight_decay=WEIGHT_DECAY)
    start = time.perf_counter()
    losses = [train_epoch(model, optimizer, data, fit_idx).item() for _ in range(best["best_epoch"])]

    torch.save(model.state_dict(), run / "model.pt")
    pd.DataFrame({"epoch": np.arange(1, len(losses) + 1), "trainval_MSE_standardized": losses}).to_csv(run / "history.csv", index=False)
    save_json(run / "completed.json", {
        "date": date, "model": name, "split_seed": split, "init_seed": seed,
        "epochs": best["best_epoch"], "trainval_nodes": int(fit_idx.numel()),
        "parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "seconds": round(time.perf_counter() - start, 2),
    })

## Run 2023 & 2024 experiments and evaluate

In [ ]:
def evaluate_date(date, x, y, out):
    rows, param_rows = [], []
    for split in SPLITS:
        masks = MASKS[split]
        data = prepare(x, y, masks["final_train_mask"])
        te_idx = mask_idx(masks["test_mask"])
        observed = y[masks["test_mask"]].astype(np.float64)
        sst = ((observed - observed.mean()) ** 2).sum()

        for name in MODELS:
            folder = out / f"split_{split}" / name
            best = json.loads((folder / "best_params.json").read_text())
            model = LSTModel(name, best["params"]).to(DEVICE)
            param_rows.append({
                "date": date, "model": name, "split_seed": split, **best["params"],
                "best_trial": best["best_trial"], "best_epoch": best["best_epoch"],
                "best_at_epoch_cap": best["best_epoch"] == MAX_EPOCHS,
                "best_val_RMSE_C": best["best_val_RMSE_C"],
                "parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
            })

            for seed in INIT_SEEDS:
                model.load_state_dict(torch.load(folder / f"init_{seed}" / "model.pt", map_location=DEVICE, weights_only=True))
                model.eval()
                with torch.no_grad():
                    pred_std = model(data)[te_idx].cpu().numpy().astype(np.float64)
                err = pred_std * data["ys"] + data["ym"] - observed
                rows.append({
                    "date": date, "model": name, "split_seed": split, "init_seed": seed,
                    "test_nodes": len(observed), "MAE": np.abs(err).mean(),
                    "RMSE": np.sqrt(np.mean(err ** 2)), "R2": 1 - np.sum(err ** 2) / sst,
                    "Bias": err.mean(),
                })

    runs = pd.DataFrame(rows)
    metrics = ["MAE", "RMSE", "R2", "Bias"]
    per_split = runs.groupby(["date", "model", "split_seed"], as_index=False)[metrics].mean()
    summary = per_split.groupby(["date", "model"])[metrics].agg(["mean", "std"])
    summary.columns = [f"{m}_{s}" for m, s in summary.columns]
    summary = summary.reset_index().sort_values("RMSE_mean")

    # Paired RMSE reduction (baseline - candidate) across the 5 splits
    pairs = [
        ("MLP_center", "MLP_ordered"),
        ("MLP_ordered", "GCN"), ("MLP_ordered", "GraphSAGE"),
        ("MLP_ordered", "GraphConv"), ("MLP_ordered", "RGCN_directional"),
        ("GCN", "RGCN_directional"), ("GraphSAGE", "RGCN_directional"),
        ("GraphConv", "RGCN_directional"),
    ]
    wide = per_split.pivot(index="split_seed", columns="model", values="RMSE")
    paired_rows = [
        {"date": date, "baseline": b, "candidate": c, "split_seed": s, "RMSE_reduction_C": wide.loc[s, b] - wide.loc[s, c]}
        for b, c in pairs for s in SPLITS
    ]
    paired = pd.DataFrame(paired_rows)
    paired_summary = paired.groupby(["date", "baseline", "candidate"], sort=False).agg(
        mean_RMSE_reduction_C=("RMSE_reduction_C", "mean"),
        SD_across_splits_C=("RMSE_reduction_C", "std"),
        candidate_better_splits=("RMSE_reduction_C", lambda v: int((v > 0).sum())),
        number_of_splits=("RMSE_reduction_C", "count"),
    ).reset_index()

    runs.to_csv(out / "test_metrics_all_runs.csv", index=False)
    per_split.to_csv(out / "comparison_per_split.csv", index=False)
    summary.to_csv(out / "comparison_summary.csv", index=False)
    pd.DataFrame(param_rows).to_csv(out / "best_params_all_splits.csv", index=False)
    paired.to_csv(out / "paired_differences_each_split.csv", index=False)
    paired_summary.to_csv(out / "paired_differences_summary.csv", index=False)

    display(summary.round(4))
    display(paired_summary.round(4))

# Main loop for 2023 and 2024
for date in RUN_DATES:
    x, y = load_date(date)
    out = WORK / f"results_grid90_10652_six_models_{date}"
    out.mkdir(parents=True, exist_ok=True)

    protocol = {
        "date": date, "graph": "90 m square grid, eight-neighbour adjacency",
        "nodes": len(y), "undirected_edges": int(E.shape[1] // 2),
        "features": FEATURES, "directions": DIRECTIONS, "models": MODELS,
        "split_seeds": SPLITS, "initializations": INIT_SEEDS,
        "trials_per_model_split": N_TRIALS, "tune_max_epochs": MAX_EPOCHS,
    }
    with open(out / "experiment_config.yml", "w", encoding="utf-8") as f:
        yaml.safe_dump(protocol, f, sort_keys=False)

    for split in SPLITS:
        masks = MASKS[split]
        tune_data = prepare(x, y, masks["train_mask"])
        final_data = prepare(x, y, masks["final_train_mask"])
        tr_idx, va_idx, fit_idx = mask_idx(masks["train_mask"]), mask_idx(masks["val_mask"]), mask_idx(masks["final_train_mask"])

        split_dir = out / f"split_{split}"
        split_dir.mkdir(exist_ok=True)
        save_json(split_dir / "tuning_scaler.json", tune_data["scaler"])
        save_json(split_dir / "final_scaler.json", final_data["scaler"])

        print(f"\n=== {date} | split={split} | train={tr_idx.numel()} | val={va_idx.numel()} | train+val={fit_idx.numel()} ===")
        for name in MODELS:
            folder = split_dir / name
            folder.mkdir(exist_ok=True)
            best = tune_six(date, name, split, tune_data, tr_idx, va_idx, folder)
            for seed in INIT_SEEDS:
                final_fit_six(date, name, split, seed, best, final_data, fit_idx, folder)

        archive = shutil.make_archive(str(out) + "_backup", "zip", root_dir=out.parent, base_dir=out.name)
        display(FileLink(Path(archive).name))

    evaluate_date(date, x, y, out)
    archive = shutil.make_archive(str(out) + "_backup", "zip", root_dir=out.parent, base_dir=out.name)
    display(FileLink(Path(archive).name))